In [5]:
import os
import pandas as pd
from bs4 import BeautifulSoup, Comment
import io

In [6]:
SCORE_DIR = "data/scores"
box_scores = os.listdir(SCORE_DIR)
box_scores = [os.path.join(SCORE_DIR,f) for f in box_scores]

In [7]:
def parse_html(box_scores):
    with open(box_scores,encoding="utf-8") as f:
        html = f.read()
    
    soup = BeautifulSoup(html)
    [s.decompose() for s in soup.select("tr.over_header")]
    [s.decompose() for s in soup.select("tr.thead")]
    return soup

In [8]:
def read_line_score(soup):
    placeholder = soup.find(id="all_line_score")
    if placeholder:
        for comment in placeholder.find_all(string=lambda text: isinstance(text, Comment)):
            if 'id="line_score"' in comment:
                table_soup = BeautifulSoup(comment, "html.parser")
                html_string = str(table_soup)
                break
    else:
        html_string = str(soup)
        
    line_score = pd.read_html(io.StringIO(html_string), attrs = {"id": "line_score"})[0]
    cols = list(line_score.columns)
    cols[0] = "team"
    cols[-1] = "total"
    line_score.columns = cols
    line_score = line_score[["team","total"]]
    return line_score

In [9]:
def read_stats(soup,team,stat):
    df = pd.read_html(io.StringIO(str(soup)),attrs={"id": f"box-{team}-game-{stat}"},index_col=0)[0]
    df = df.apply(pd.to_numeric, errors="coerce")
    return df

In [10]:
def get_season_info(soup):
    nav = soup.select("#bottom_nav_container")[0]
    hrefs = [a["href"] for a in nav.find_all("a")]
    season = os.path.basename(hrefs[1]).split("_")[0]
    return int(season)

In [11]:
base_cols = None
games = []
for box_score in box_scores:
    soup = parse_html(box_score)
    line_score = read_line_score(soup)
    teams = list(line_score["team"])

    summaries = []
    for i,team in enumerate(teams):
        basic = read_stats(soup,team,"basic")
        advanced = read_stats(soup,team,"advanced")

        totals = pd.concat([basic.iloc[-1,:], advanced.iloc[-1,:]])
        totals.index = totals.index.str.lower()
        totals = totals.rename(i)
        if base_cols is None:
            base_cols = list(totals.index)
            base_cols = [b for b in base_cols if "bpm" not in b]
        
        summary = totals[base_cols]
        summaries.append(summary)

    summary = pd.concat(summaries, axis=1).T

    game = pd.concat([summary,line_score],axis=1)
    game = game.loc[:, ~game.columns.duplicated()]

    game["home"] = [0,1]
    game_opp = game.iloc[::-1].reset_index()
    game_opp.columns += "_opp"

    full_game = pd.concat([game,game_opp],axis=1)

    full_game["season"] = get_season_info(soup)

    full_game["date"] = os.path.basename(box_score)[:8]
    full_game["date"] = pd.to_datetime(full_game["date"], format="%Y%m%d")

    full_game["won"] = full_game["total"] > full_game["total_opp"]

    full_game["id"] = os.path.basename(box_score)[:-5]

    games.append(full_game)

    if len(games) % 500 == 0:
        print(f"{len(games)} / {len(box_scores)} ")

500 / 13666 
1000 / 13666 
1500 / 13666 
2000 / 13666 
2500 / 13666 
3000 / 13666 
3500 / 13666 
4000 / 13666 
4500 / 13666 
5000 / 13666 
5500 / 13666 
6000 / 13666 
6500 / 13666 
7000 / 13666 
7500 / 13666 
8000 / 13666 
8500 / 13666 
9000 / 13666 
9500 / 13666 
10000 / 13666 
10500 / 13666 
11000 / 13666 
11500 / 13666 
12000 / 13666 
12500 / 13666 
13000 / 13666 
13500 / 13666 


In [12]:
games_df = pd.concat(games,ignore_index=True)

In [13]:
games_df

,mp,fg,fga,fg%,3p,3pa,3p%,ft,fta,ft%,...,usg%_opp,ortg_opp,drtg_opp,team_opp,total_opp,home_opp,season,date,won,id
0,240.0,37.0,96.0,0.385,12.0,29.0,0.414,20.0,26.0,0.769,...,100.0,98.6,111.2,ATL,94,1,2016,2015-10-27,True,201510270ATL
1,240.0,37.0,82.0,0.451,8.0,27.0,0.296,12.0,15.0,0.800,...,100.0,111.2,98.6,DET,106,0,2016,2015-10-27,False,201510270ATL
2,240.0,38.0,94.0,0.404,9.0,29.0,0.310,10.0,17.0,0.588,...,100.0,97.0,95.0,CHI,97,1,2016,2015-10-27,False,201510270CHI
3,240.0,37.0,87.0,0.425,7.0,19.0,0.368,16.0,23.0,0.696,...,100.0,95.0,97.0,CLE,95,0,2016,2015-10-27,True,201510270CHI
4,240.0,35.0,83.0,0.422,6.0,18.0,0.333,19.0,27.0,0.704,...,100.0,110.3,94.4,GSW,111,1,2016,2015-10-27,False,201510270GSW
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27327,240.0,48.0,86.0,0.558,10.0,31.0,0.323,18.0,21.0,0.857,...,100.0,106.7,127.2,DAL,104,0,2026,2026-02-12,True,202602120LAL
27328,240.0,44.0,95.0,0.463,17.0,42.0,0.405,5.0,7.0,0.714,...,100.0,101.9,120.5,OKC,93,1,2026,2026-02-12,True,202602120OKC
27329,240.0,31.0,83.0,0.373,15.0,45.0,0.333,16.0,20.0,0.800,...,100.0,120.5,101.9,MIL,110,0,2026,2026-02-12,False,202602120OKC
27330,240.0,46.0,90.0,0.511,18.0,46.0,0.391,25.0,33.0,0.758,...,100.0,113.8,129.1,UTA,119,1,2026,2026-02-12,True,202602120UTA


In [14]:
games_df.to_csv("nba_games.csv")